# Training Report — Plant Disease Detection (v2)
**Final Project Pembelajaran Mesin 2** — Farrel Ghozy Affifudin (452024611053) — TI5 A2
**Universitas Darussalam Gontor**

Eksperimen deep learning klasifikasi penyakit tanaman pada **dataset PlantVillage** (38 kelas, 54.305 citra),
dijalankan pada **NVIDIA RTX 4060 (CUDA 12.4, mixed precision FP16)**.

| Exp | Model | Strategi | LR | Epoch |
|-----|-------|----------|-----|-------|
| E1 | Custom CNN (0,54M) | Dari nol (baseline) | 1e-3 | 15 |
| E2 | MobileNetV3-Small | Transfer learning, frozen | 1e-3 | 12 |
| E3 | EfficientNet-B0 | Transfer learning, frozen | 1e-3 | 12 |
| E4 | MobileNetV3-Small | Transfer learning, fine-tune | 1e-4 | 12 |

Notebook ini membaca hasil eksperimen nyata dari folder `results/` (metrics, kurva training, confusion matrix, error analysis).


In [ ]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

RESULTS = 'results'  # folder relatif terhadap repo root
rows = []
for exp in sorted(glob.glob(f'{RESULTS}/e*')):
    name = os.path.basename(exp)
    m = json.load(open(f'{exp}/metrics.json'))
    t = m['test']
    rows.append({
        'Exp': name.upper(), 'Model': m['model'], 'Strategy': m['description'],
        'Acc': round(t['accuracy'], 4), 'Prec': round(t['precision_macro'], 4),
        'Rec': round(t['recall_macro'], 4), 'F1': round(t['f1_macro'], 4),
        'Epoch': m['config']['best_epoch'],
        'TrainTime(s)': round(m['train_seconds']),
        'ParamsTrainable': f"{m['params']['trainable']:,}",
    })
df = pd.DataFrame(rows)
display(df.style.highlight_max(subset=['Acc', 'Prec', 'Rec', 'F1'], color='#c8e6c9'))
print('\nRata-rata waktu per epoch (detik):')
df['sec/epoch'] = df['TrainTime(s)'] / df['Epoch']
print(df[['Exp', 'sec/epoch']].to_string(index=False))

## Kurva Training & Validasi
Gambar di bawah adalah kurva loss dan accuracy dari proses training nyata (bukan simulasi).


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
exps = ['e1','e2','e3','e4']
for i, e in enumerate(exps):
    path = f'{RESULTS}/{e}/training_curves.png'
    if not os.path.exists(path):
        axes[0,i].set_title(f'{e.upper()} (belum ada)')
        continue
    img = mpimg.imread(path)
    axes[0,i].imshow(img); axes[0,i].axis('off'); axes[0,i].set_title(e.upper())
fig.suptitle('Training Curves — Semua Eksperimen', fontsize=14)
plt.tight_layout(); plt.show()

## Confusion Matrix (Test Set)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(28, 9))
for i, e in enumerate(exps):
    path = f'{RESULTS}/{e}/confusion_matrix.png'
    if not os.path.exists(path):
        axes[i].set_title(f'{e.upper()} (belum ada)')
        continue
    img = mpimg.imread(path)
    axes[i].imshow(img); axes[i].axis('off'); axes[i].set_title(e.upper())
plt.tight_layout(); plt.show()

## Error Analysis
### Akurasi per kelas — 10 terbaik & 10 terburuk (model terbaik)


In [ ]:
best = df.sort_values('F1', ascending=False).iloc[0]['Exp'].lower()
info = json.load(open(f'{RESULTS}/{best}/per_class_analysis.json'))
print(f'=== Model terbaik: {best.upper()} (F1={df.sort_values("F1", ascending=False).iloc[0]["F1"]}) ===')
print('\n-- 10 kelas TERBURUK --')
for x in info['worst']: print(f"  {x['class']}: {x['acc']:.4f}")
print('\n-- 10 kelas TERBAIK --')
for x in info['best']: print(f"  {x['class']}: {x['acc']:.4f}")

### Top-25 kesalahan dengan confidence tertinggi (model salah tapi yakin)
Ini sampel misclassification yang paling "yakin" — berguna untuk memahami pola kesalahan model.


In [ ]:
errs = json.load(open(f'{RESULTS}/{best}/top_errors.json'))
for i, e in enumerate(errs[:10], 1):
    print(f"{i:2d}. {os.path.basename(e['path'])[:55]:55s} true={e['true'][:30]:30s} pred={e['pred'][:30]:30s} conf={e['confidence']:.3f}")

## Classification Report (per kelas, model terbaik)


In [ ]:
print(open(f'{RESULTS}/{best}/classification_report.txt').read())

---
*Seluruh hasil di atas bersumber dari eksperimen nyata yang dijalankan di GPU lokal (RTX 4060).
Log training lengkap: `results/run_all.log`. Checkpoint model: `~/pm2_v2/models/<exp>_best.pt`.*